In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab

Mounted at /content/drive
/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab


In [2]:
import pandas as pd
from pathlib import Path

In [3]:
data_info_file = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/dataset-raw/data.xlsx"
dt = pd.read_excel(data_info_file)
dt.head()

,health center,id,sex,age_month,weight,height,measurement,haz score,state,availability,state_
0,health center 1,1,2.0,42.0,11.5,89.7,standing,-2.30,Stunting,1,1
1,health center 1,2,2.0,59.0,12.3,94.6,standing,-3.03,Severe stunting,1,1
2,health center 1,3,2.0,24.0,9.2,78.5,standing,-2.24,Stunting,1,1
3,health center 1,4,1.0,33.0,11.7,86.5,standing,-2.12,Stunting,1,1
4,health center 1,5,1.0,57.0,13.0,92.6,standing,-3.47,Severe stunting,1,1


In [4]:
# Pipeline: List of Functions
HEALTH_CENTER_INFO = {
    "health center 1": {
        "folder": "health-center-1",
        "code": "gnd"
    },
    "health center 2": {
        "folder": "health-center-2",
        "code": "kry"
    },
    "health center 3": {
        "folder": "health-center-3",
        "code": "ktp"
    },
}

def load_metadata(base_dir, file_name):
    return pd.read_excel(base_dir / file_name).dropna()

def select_available_standing(df):
    df = df.copy()

    df = df[
        (df["measurement"] == "standing") &
        (df["availability"] == 1)
    ]

    return df.reset_index(drop=True)

def add_basic_columns(df):
    df = df.copy().reset_index(drop=True)

    df["child_id"]                  = df.index
    df["child_id_at_health_center"] = df["id"]
    df["health_center"]             = df["health center"]
    df["label"]                     = df["state_"].astype(int)
    df["cap_kind"]                  = "original"

    return df

def add_image_path(df, base_dir, health_center_info=HEALTH_CENTER_INFO):
    df = df.copy()

    def make_path(row):
        info = health_center_info[row["health_center"]]

        return (
            base_dir
            / info["folder"]
            / f"{row['child_id_at_health_center']}_wajah_crop_{info['code']}.jpg"
        )

    df["path"] = df.apply(make_path, axis=1)

    return df

def keep_existing_files(df):
    df = df.copy()

    return (
        df[df["path"].apply(lambda p: Path(p).exists())]
        .reset_index(drop=True)
    )

def select_df_files_columns(df):
    return (
        df[
            [
                "child_id",
                "state",
                "cap_kind",
                "label",
                "health_center",
                "path"
            ]
        ]
        .assign(path=lambda df: df["path"].astype(str))
        .reset_index(drop=True)
    )

In [5]:
BASE_DIR    = Path("/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/dataset-raw/")
FILE_NAME   = "data.xlsx"

dataset_df = (
    # Data Collection
    load_metadata(BASE_DIR, FILE_NAME)
    .pipe(select_available_standing)
    .pipe(add_basic_columns)
    .pipe(add_image_path, base_dir=BASE_DIR)
    .pipe(keep_existing_files)
    .pipe(select_df_files_columns)
)

display(dataset_df)
dataset_df.to_excel('dataset_info.xlsx', index=False)

,child_id,state,cap_kind,label,health_center,path
0,0,Stunting,original,1,health center 1,/content/drive/MyDrive/Stunted Children Identi...
1,1,Severe stunting,original,1,health center 1,/content/drive/MyDrive/Stunted Children Identi...
2,2,Stunting,original,1,health center 1,/content/drive/MyDrive/Stunted Children Identi...
3,3,Stunting,original,1,health center 1,/content/drive/MyDrive/Stunted Children Identi...
4,4,Severe stunting,original,1,health center 1,/content/drive/MyDrive/Stunted Children Identi...
...,...,...,...,...,...,...
185,185,Stunting,original,1,health center 2,/content/drive/MyDrive/Stunted Children Identi...
186,186,Severe stunting,original,1,health center 2,/content/drive/MyDrive/Stunted Children Identi...
187,187,Stunting,original,1,health center 2,/content/drive/MyDrive/Stunted Children Identi...
188,188,Stunting,original,1,health center 2,/content/drive/MyDrive/Stunted Children Identi...


In [6]:
import os
os.environ["PYTHONHASHSEED"] = "123"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # must be before torch import

In [7]:
import math, random, hashlib, copy
import pandas as pd
import numpy as np

from pathlib import Path
from typing  import Tuple, List
from PIL     import Image, ImageEnhance

from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF

from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from sklearn.metrics import fbeta_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

In [8]:
dataset_df = pd.read_excel("/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/dataset_info.xlsx")
dataset_df.head()

,child_id,state,cap_kind,label,health_center,path
0,0,Stunting,original,1,health center 1,/content/drive/MyDrive/Stunted Children Identi...
1,1,Severe stunting,original,1,health center 1,/content/drive/MyDrive/Stunted Children Identi...
2,2,Stunting,original,1,health center 1,/content/drive/MyDrive/Stunted Children Identi...
3,3,Stunting,original,1,health center 1,/content/drive/MyDrive/Stunted Children Identi...
4,4,Severe stunting,original,1,health center 1,/content/drive/MyDrive/Stunted Children Identi...


In [9]:
# Random Configuration

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

torch.use_deterministic_algorithms(True)  # ← add here

In [10]:
# -----------------------------
# Deterministic RNG per (child, cap, stage, aug)
# -----------------------------
def make_rng(*parts, base_seed=123):
    s = str(base_seed) + "|" + "|".join(map(str, parts))
    seed = (abs(hash(s)) % (2**32 - 1))
    return random.Random(seed), np.random.default_rng(seed)


#PSEUDO CAPTURE

In [11]:
# Pseudo-capture Transformation Function

# -----------------------------
# Deterministic RNG per (child, cap, stage, aug)
# -----------------------------
#def make_rng(*parts, base_seed=123):
#    s = str(base_seed) + "|" + "|".join(map(str, parts))
#    seed = (abs(hash(s)) % (2**32 - 1))
#    return random.Random(seed), np.random.default_rng(seed)

"""
# the same child, capture index, and seed will always generate the same pseudo-capture

def make_rng(*parts, base_seed=123):
    s = str(base_seed) + "|" + "|".join(map(str, parts))
    seed = int(hashlib.sha256(s.encode("utf-8")).hexdigest(), 16) % (2**32 - 1)
    return random.Random(seed), np.random.default_rng(seed)
"""

# -----------------------------
# Gentle pseudo-capture transforms
# -----------------------------
def apply_pseudo_capture_geometry(
    img: Image.Image,
    rng: random.Random,
    rot_deg: float = 5.0,
    translate_frac: float = 0.03,
    scale_range: Tuple[float, float] = (0.97, 1.03),
) -> Image.Image:
    w, h = img.size
    angle = rng.uniform(-rot_deg, rot_deg)

    max_dx = translate_frac * w
    max_dy = translate_frac * h
    translate = (int(rng.uniform(-max_dx, max_dx)), int(rng.uniform(-max_dy, max_dy)))

    scale = rng.uniform(scale_range[0], scale_range[1])

    shear = [0.0, 0.0]

    return TF.affine(
        img,
        angle=angle,
        translate=translate,
        scale=scale,
        shear=shear,
        interpolation=TF.InterpolationMode.BICUBIC,
        fill=0
    )

def apply_pseudo_capture_horizontal_flip(
    img: Image.Image,
    rng: random.Random,
    p: float = 0.3
) -> Image.Image:
    return TF.hflip(img) if rng.random() < p else img


def apply_pseudo_capture_brightness_contrast(
    img: Image.Image,
    rng: random.Random,
    brightness: float = 0.08,
    contrast: float = 0.08
) -> Image.Image:
    if brightness > 0:
        b = 1.0 + rng.uniform(-brightness, brightness)
        img = ImageEnhance.Brightness(img).enhance(b)
    if contrast > 0:
        c = 1.0 + rng.uniform(-contrast, contrast)
        img = ImageEnhance.Contrast(img).enhance(c)
    return img

In [12]:
# Pseudo-capture Implementation Function

def pseudo_capture_image(df, out_dir, cfg):

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    rows = []

    for i in range(len(df)):
        src_path = str(df.iloc[i]["path"])
        label = int(df.iloc[i]["label"])
        child_id = str(df.iloc[i]["child_id"])

        base_img = Image.open(src_path).convert("RGB")

        for cap in range(cfg.n_captures_total):
            if cap == 0:
                img_cap = base_img.copy()
                cap_kind = "original"
            else:
                rng_cap, _ = make_rng(child_id, "cap", cap, "pseudo_capture", base_seed=cfg.base_seed)
                img_cap    = base_img.copy()

                img_cap = apply_pseudo_capture_geometry(
                    img_cap, rng_cap,
                    rot_deg=cfg.rot_deg,
                    translate_frac=cfg.translate_frac,
                    scale_range=cfg.scale_range,
                )

                img_cap = apply_pseudo_capture_horizontal_flip(img_cap, rng_cap, p=cfg.flip_p)

                img_cap = apply_pseudo_capture_brightness_contrast(
                    img_cap, rng_cap,
                    brightness=cfg.brightness,
                    contrast=cfg.contrast
                )

                cap_kind = "original-pseudo"

            # Save base capture
            fname_base = f"child_{child_id}_label_{label}_cap{cap:02d}_base.jpg"
            fpath_base = out_dir / fname_base
            img_cap.save(fpath_base, quality=95)

            rows.append({
                "child_id": child_id,
                "cap_id": cap,
                "stage": "capture",
                "variant": "base_capture",
                "cap_kind": cap_kind,
                "label": label,
                "source_path": src_path,
                "path": str(fpath_base),
            })

    return pd.DataFrame(rows)

In [13]:
# Pseudo Configuration

@dataclass
class PseudoCaptureConfig:
    n_captures_total: int = 10
    base_seed: int = SEED

    rot_deg: float = 5.0
    translate_frac: float = 0.03
    scale_range: Tuple[float, float] = (0.97, 1.03)
    flip_p: float = 0.3
    brightness: float = 0.08
    contrast: float = 0.08

In [14]:
# Select columns

def select_pseudo_capture_columns(df):
    return df[["path", "label", "child_id"]].copy()

In [15]:
OUT_DIR   = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/pseudo-capture-images"

pseudo_capture_dataset_df = (
    dataset_df
    .pipe(select_pseudo_capture_columns)
    .pipe(
        pseudo_capture_image,
        out_dir=OUT_DIR,
        cfg=PseudoCaptureConfig()
        )
    )

display(pseudo_capture_dataset_df)
pseudo_capture_dataset_df.to_excel('pseudo_capture_dataset_info.xlsx', index=False)

,child_id,cap_id,stage,variant,cap_kind,label,source_path,path
0,0,0,capture,base_capture,original,1,/content/drive/MyDrive/Stunted Children Identi...,/content/drive/MyDrive/Stunted Children Identi...
1,0,1,capture,base_capture,original-pseudo,1,/content/drive/MyDrive/Stunted Children Identi...,/content/drive/MyDrive/Stunted Children Identi...
2,0,2,capture,base_capture,original-pseudo,1,/content/drive/MyDrive/Stunted Children Identi...,/content/drive/MyDrive/Stunted Children Identi...
3,0,3,capture,base_capture,original-pseudo,1,/content/drive/MyDrive/Stunted Children Identi...,/content/drive/MyDrive/Stunted Children Identi...
4,0,4,capture,base_capture,original-pseudo,1,/content/drive/MyDrive/Stunted Children Identi...,/content/drive/MyDrive/Stunted Children Identi...
...,...,...,...,...,...,...,...,...
1895,189,5,capture,base_capture,original-pseudo,0,/content/drive/MyDrive/Stunted Children Identi...,/content/drive/MyDrive/Stunted Children Identi...
1896,189,6,capture,base_capture,original-pseudo,0,/content/drive/MyDrive/Stunted Children Identi...,/content/drive/MyDrive/Stunted Children Identi...
1897,189,7,capture,base_capture,original-pseudo,0,/content/drive/MyDrive/Stunted Children Identi...,/content/drive/MyDrive/Stunted Children Identi...
1898,189,8,capture,base_capture,original-pseudo,0,/content/drive/MyDrive/Stunted Children Identi...,/content/drive/MyDrive/Stunted Children Identi...


#REFERENCE BASED AUGMENTATION

In [16]:
# Reference based Augmentation Transformation Function

def ref_zoom_in(img: Image.Image, zoom_factor: float = 1.10) -> Image.Image:
    # Center crop after scaling up to keep same size.
    w, h         = img.size
    new_w, new_h = int(round(w * zoom_factor)), int(round(h * zoom_factor))
    scaled       = img.resize((new_w, new_h), Image.BICUBIC)
    left         = (new_w - w) // 2
    top          = (new_h - h) // 2
    return scaled.crop((left, top, left + w, top + h))

def ref_rotation_one_angle(img: Image.Image, rng: random.Random, angles=(10, 15, 20)) -> Tuple[Image.Image, int]:
    ang = rng.choice(list(angles))  # random but deterministic because rng is seeded
    out = TF.rotate(img, angle=ang, interpolation=TF.InterpolationMode.BICUBIC, fill=0)
    return out, ang

def ref_translate_right(img: Image.Image, frac_right: float = 0.10) -> Image.Image:
    w, h = img.size
    dx   = int(round(frac_right * w))
    return TF.affine(
        img,
        angle=0.0,
        translate=(dx, 0),
        scale=1.0,
        shear=[0.0, 0.0],
        interpolation=TF.InterpolationMode.BICUBIC,
        fill=0
    )

In [17]:
# Reference based Augmentation Configuration

@dataclass
class RefAugmentationConfig:
    base_seed: int = SEED
    zoom_factor: float = 1.10
    rotation_angles: Tuple[int, ...] = (10, 15, 20)
    shift_right_frac: float = 0.10

In [21]:
# Reference-based Augmentation Implementation Function

def ref_augmentated_image(df, out_dir, cfg):

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []

    for idx, row in df.iterrows():
        src_path = str(row["path"])
        label    = int(row["label"])
        child_id = str(row["child_id"])


        cap      = int(row["cap_id"])
        cap_kind = str(row["cap_kind"])
        path     = str(row["path"])

        rows.append({
            "child_id"   : child_id,
            "cap_id"     : cap,
            "stage"      : "capture",
            "variant"    : "base_capture",
            "cap_kind"   : cap_kind,
            "label"      : label,
            "source_path": src_path,
            "path"       : path,
            })

        base_img = Image.open(path).convert("RGB")

        # Zoom-in
        img_zoom   = ref_zoom_in(base_img, zoom_factor=cfg.zoom_factor)
        fname_zoom = f"child_{child_id}_label_{label}_cap{cap:02d}_paper_zoom.jpg"
        fpath_zoom = out_dir / fname_zoom
        img_zoom.save(fpath_zoom, quality=95)

        rows.append({
            "child_id"    : child_id,
            "cap_id"      : cap,
            "stage"       : "ref_augmentation",
            "variant"     : "zoom_in",
            "cap_kind"    : cap_kind + "-" + "augmentation",
            "label"       : label,
            "source_path" : src_path,
            "path"        : str(fpath_zoom),
        })

        # Rotation: randomly pick ONE angle from {10,15,20} deterministically
        rng_rot, _   = make_rng(child_id, "cap", cap, "paper_rotation", base_seed=cfg.base_seed)
        img_rot, ang = ref_rotation_one_angle(base_img, rng_rot, angles=cfg.rotation_angles)
        fname_rot    = f"child_{child_id}_label_{label}_cap{cap:02d}_paper_rot{ang}.jpg"
        fpath_rot    = out_dir / fname_rot
        img_rot.save(fpath_rot, quality=95)

        rows.append({
            "child_id"   : child_id,
            "cap_id"     : cap,
            "stage"      : "ref_augmentation",
            "variant"    : "rotation",
            "cap_kind"   : cap_kind + "-" + "augmentation",
            "label"      : label,
            "source_path": src_path,
            "path": str(fpath_rot),
        })

        # (Right translation (10% width)
        img_shift   = ref_translate_right(base_img, frac_right=cfg.shift_right_frac)
        fname_shift = f"child_{child_id}_label_{label}_cap{cap:02d}_paper_shiftR10.jpg"
        fpath_shift = out_dir / fname_shift
        img_shift.save(fpath_shift, quality=95)

        rows.append({
            "child_id"   : child_id,
            "cap_id"     : cap,
            "stage"      : "ref_augmentation",
            "variant"    : "shift_right",
            "cap_kind"   : cap_kind + "-" + "augmentation",
            "label"      : label,
            "source_path": src_path,
            "path"       : str(fpath_shift),
        })

    return pd.DataFrame(rows)

In [22]:
OUT_DIR_AUG = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/ref-based-augmentation-images/"

ref_based_aug_dataset_df = (
    pseudo_capture_dataset_df
    .pipe(
        ref_augmentated_image,
        out_dir=OUT_DIR_AUG,
        cfg=RefAugmentationConfig(),
        )
    )

display(ref_based_aug_dataset_df)
ref_based_aug_dataset_df.to_excel('ref_based_aug_dataset_info.xlsx', index=False)

,child_id,cap_id,stage,variant,cap_kind,label,source_path,path
0,0,0,capture,base_capture,original,1,/content/drive/MyDrive/Stunted Children Identi...,/content/drive/MyDrive/Stunted Children Identi...
1,0,0,ref_augmentation,zoom_in,original-augmentation,1,/content/drive/MyDrive/Stunted Children Identi...,/content/drive/MyDrive/Stunted Children Identi...
2,0,0,ref_augmentation,rotation,original-augmentation,1,/content/drive/MyDrive/Stunted Children Identi...,/content/drive/MyDrive/Stunted Children Identi...
3,0,0,ref_augmentation,shift_right,original-augmentation,1,/content/drive/MyDrive/Stunted Children Identi...,/content/drive/MyDrive/Stunted Children Identi...
4,0,1,capture,base_capture,original-pseudo,1,/content/drive/MyDrive/Stunted Children Identi...,/content/drive/MyDrive/Stunted Children Identi...
...,...,...,...,...,...,...,...,...
7595,189,8,ref_augmentation,shift_right,original-pseudo-augmentation,0,/content/drive/MyDrive/Stunted Children Identi...,/content/drive/MyDrive/Stunted Children Identi...
7596,189,9,capture,base_capture,original-pseudo,0,/content/drive/MyDrive/Stunted Children Identi...,/content/drive/MyDrive/Stunted Children Identi...
7597,189,9,ref_augmentation,zoom_in,original-pseudo-augmentation,0,/content/drive/MyDrive/Stunted Children Identi...,/content/drive/MyDrive/Stunted Children Identi...
7598,189,9,ref_augmentation,rotation,original-pseudo-augmentation,0,/content/drive/MyDrive/Stunted Children Identi...,/content/drive/MyDrive/Stunted Children Identi...


In [23]:
ref_based_aug_dataset_df.cap_kind.unique()

array(['original', 'original-augmentation', 'original-pseudo',
       'original-pseudo-augmentation'], dtype=object)